<div dir="rtl" style="text-align:right">
<h1>از نویسه تا نمایش یادگرفتنی</h1><p style="text-align:right"><b>پرسش آزمایش:</b> شناسه، سطر Embedding و موقعیت چه فرق‌هایی دارند؟</p><p style="text-align:right">پیش‌نیاز: <a href="http://127.0.0.1:8000/part-04/chapter-01/21-tokenizer.html"><bdi dir="ltr">21-tokenizer</bdi></a>، <a href="http://127.0.0.1:8000/part-04/chapter-02/23-shift.html"><bdi dir="ltr">23-shift</bdi></a>، <a href="http://127.0.0.1:8000/part-04/chapter-03/25-embedding.html"><bdi dir="ltr">25-embedding</bdi></a>، <a href="http://127.0.0.1:8000/part-04/chapter-03/26-positions.html"><bdi dir="ltr">26-positions</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای تکرار پاک، Kernel را Restart و سپس Run All کنید. لینک درس با سروکردن کتاب روی پورت ۸۰۰۰ کار می‌کند؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">از Tokenizer و Dataset واقعی پروژه استفاده می‌کنیم. جدول نمایش این آزمایش کوچک و تصادفی است؛ از آن دربارهٔ معنای واژه‌ها نتیجه نمی‌گیریم. واژگان فقط از متن مرجعِ تعیین‌شده ساخته می‌شود.</p>
</div>

In [ ]:
from torch import nn
from mini_gpt.tokenizer import CharacterTokenizer
from mini_gpt.dataset import NextTokenDataset
reference = "مدل می‌رود. مدل می‌آید."
tokenizer = CharacterTokenizer.from_text(reference)
print(list(enumerate(tokenizer.id_to_token)))
ids = tokenizer.encode(reference)
dataset = NextTokenDataset(ids, context_length=6)
x, y = dataset[0]
print("input:", x.tolist(), tokenizer.decode(x.tolist()))
print("target:", y.tolist(), tokenizer.decode(y.tolist()))
assert torch.equal(x[1:], y[:-1])
inspect("token IDs", x)
print("Unknown example:", tokenizer.encode("🐈"), tokenizer.decode(tokenizer.encode("🐈")))


<div dir="rtl" style="text-align:right">
<p style="text-align:right">پیش‌بینی کنید دو بار آمدن یک شناسه، پیش از افزودن موقعیت چه خروجی می‌دهد. آیا بزرگ‌تر بودن شناسه به معنی بزرگ‌تر بودن ویژگی‌هاست؟</p>
</div>

In [ ]:
C = 4
embedding = nn.Embedding(tokenizer.vocab_size, C)
position = nn.Embedding(6, C)
same_ids = torch.tensor([[1,1,2]], dtype=torch.long)
tokens = embedding(same_ids)                       # (B,T,C)
positions = position(torch.arange(same_ids.shape[1]))  # (T,C)
combined = tokens + positions
for name, value in [("IDs",same_ids),("Token Embedding",tokens),
                    ("Position Embedding",positions),("Combined",combined)]:
    inspect(name, value)
    print(value.detach())
torch.testing.assert_close(tokens[0,0], tokens[0,1])
assert not torch.equal(combined[0,0], combined[0,1])
one_hot = torch.nn.functional.one_hot(same_ids, tokenizer.vocab_size).float()
torch.testing.assert_close(one_hot @ embedding.weight, tokens)


<div dir="rtl" style="text-align:right">
<h2>کدام عددها Gradient می‌گیرند؟</h2><p style="text-align:right">شناسه‌ها عدد صحیح و نشانی سطرند؛ پارامترهای جدول یاد می‌گیرند. Loss زیر فقط مجموع عددهاست تا مسیر مشتق دیده شود، نه هدف آموزش زبان. حدس بزنید چرا سطر شناسهٔ ۱ دو برابر سهم می‌گیرد.</p>
</div>

In [ ]:
embedding.zero_grad(set_to_none=True)
embedding(same_ids).sum().backward()
inspect("Embedding Gradient", embedding.weight.grad)
print(embedding.weight.grad)
torch.testing.assert_close(embedding.weight.grad[1], torch.full((C,),2.))
torch.testing.assert_close(embedding.weight.grad[2], torch.ones(C))
assert same_ids.grad is None
try:
    embedding(torch.tensor([tokenizer.vocab_size]))
except (IndexError, RuntimeError) as error:
    print("Expected out-of-vocabulary index:", error)
else:
    raise AssertionError("Expected invalid index")


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین:</b> Tokenizer فعلی را ثابت نگه دارید. نسخهٔ تازه‌ای از متن بسازید و در آن «ی» را با «ي» یا فاصله را با نیم‌فاصله عوض کنید؛ همان Tokenizer را روی ورودی تازه اجرا کنید. انتظار دارید طول، شناسه‌ها و نرخ ناشناخته چگونه تغییر کند؟ سپس context_length را تغییر دهید و زوج ورودی/هدف تازه را دستی بررسی کنید. نمایش Token همراه موقعیت هنوز ارتباطِ آموخته میان موقعیت‌ها نیست؛ این کار را در Attention می‌سازیم.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2>برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی، مشاهده و دلیل اختلافشان را در یادداشت خود بنویسید. سپس به <a href="http://127.0.0.1:8000/part-04/chapter-03/26-positions.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>